**Cell 1**# NB09 — Reward-Free Geometry of the RS-PPO AdaptersComputes the geometry of the five RS-PPO LoRA specialists produced by NB08. This notebook is the **reward-free** half of the analysis pipeline: it never loads ArmoRM and never issues a reward query. Everything here is a statement about where the five task vectors sit relative to each other, independent of how they were trained.**Scope — what this notebook delivers.**| Step | Object | Deliverable ||---|---|---|| D | $D_i = \theta_i - \theta_{\mathrm{SFT}}$, exact effective LoRA update | prerequisite || R | $R = D^\top D$ (Gram) and $R_{\cos}$ | prerequisite || Floor-LP, $R^-$ | is the improvement floor $\mathcal F_p$ open? | **(b)** || LMC | loss barriers along interpolation paths | **(d)** |**What this notebook does NOT do.** No ArmoRM, no search set scoring, no vertex matrix $M$, no $U_p(\lambda)$, no method comparison. Those belong to deliverable (a), which is currently blocked pending the §6 escalation. Deliverables (b) and (d) are purely geometric and are unaffected by that escalation — this is exactly why they run first.**Why not PEFT.** `peft.add_weighted_adapter(..., combination_type="linear")` does **not** realise $\theta_0 + \sum_i \lambda_i \delta_i$: it combines the factors under a square root and the product $B_{\mathrm{merged}}A_{\mathrm{merged}}$ picks up cross terms $B_iA_j$ for every interior $\lambda$ (measured relative error 1.47). All merging in this notebook goes through `src/merge.py`, which accumulates the effective deltas directly. Cell 25 verifies this end-to-end against an independent oracle **and** against a deliberately contaminated control that must fail.**Precision.** Every merge and every evaluation forward pass runs in **float32**. bf16 storage injects rounding noise comparable to the endpoint displacement itself (noise/signal 1.64) and destroys the primary endpoint. `MERGE_DTYPE = "float32"` is asserted by static checks S9/S10 before any geometry is computed.**Provisional numbers.** Every floor / $R^-$ / Perron figure currently in the thesis comes from the **v5 SFT-R** and is provisional. This notebook replaces them. Do not assume the floor collapse reproduces — it may or may not.

**Cell 2**## 1. Clone or update the repository

In [ ]:
# Cell 3%cd /contentimport os, shutilrepo_path = "/content/master-thesis"repo_url = "https://github.com/NZhang137/master-thesis.git"if os.path.isdir(os.path.join(repo_path, ".git")):    %cd /content/master-thesis    !git pullelse:    if os.path.exists(repo_path):        shutil.rmtree(repo_path)    !git clone {repo_url} {repo_path}    %cd /content/master-thesis

**Cell 4**## 2. Check the machineUnlike NB08 this notebook is mostly **RAM**-bound, not VRAM-bound. The effective delta $B_iA_i$ of one adapter materialises to roughly the full target-weight footprint (~0.97 B parameters for TinyLlama's seven LoRA modules), i.e. ~3.9 GB in fp32 per adapter. R is therefore computed **pairwise streaming** — never more than two adapters resident at once (~8 GB peak) — rather than by holding all five.A GPU is only needed for the LMC phase (forward passes in fp32). Geometry runs fine on CPU.

In [ ]:
# Cell 5!nvidia-smi || echo "No GPU visible — geometry still runs, LMC does not."import psutilprint(f"\nRAM total {psutil.virtual_memory().total/1e9:.1f} GB  "      f"available {psutil.virtual_memory().available/1e9:.1f} GB")print("Rule of thumb: pairwise streaming needs ~8 GB free. Below that, lower LMC_BATCH first.")

**Cell 6**## 3. Install dependenciesSame pinned set as NB08 minus `trl` and `bitsandbytes` (no PPO, no quantised reward model), plus `scipy` for the floor LP. The Transformers / PEFT / Accelerate pins stay because the adapters were written by PEFT 0.10.0 and θ_SFT must be reconstructed by the same loader that produced it in NB08.

In [ ]:
# Cell 7!pip uninstall -y torchao!pip install -q -U "pandas==2.2.2" "numpy<2.1" "protobuf>=5.29.1,<6.0.0" "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" scipy datasets pyyaml safetensors psutil

**Cell 8**Restart the runtime once after installation so Python forgets previously imported Transformers or PyTorch modules, then rerun the repository cell (Cell 3) and continue.

**Cell 9**## 4. Settings and gates`RUN_*` flags mirror NB08's gate structure: **each flag is only switched on after the previous phase has written a passing entry into `report.json`.** Do not enable them all at once — the whole point is that a failed gate stops the pipeline before it contaminates the next stage.Gate order: `RESTORE → STATIC → GEOMETRY → FLOOR → LMC`.`ADAPTER_ZIP` and `ADAPTER_ZIP_SHA256` point at the NB08 bundle in Drive. The SHA is what makes the chain auditable: it is the evidence that the adapters R was computed from are the adapters PPO produced.

In [ ]:
# Cell 10import osfrom pathlib import PathBASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"RUN_TAG      = "run1"# --- inputs -----------------------------------------------------------------ADAPTER_ZIP        = "/content/drive/MyDrive/rs_ppo_armorm_adapters.zip"ADAPTER_SHA_FILE   = "/content/drive/MyDrive/rs_ppo_armorm_adapters.sha256"THETA_SFT_PROV     = "/content/drive/MyDrive/theta_sft_provenance.txt"   # optional revision pin# --- outputs ----------------------------------------------------------------WORK_DIR   = Path(f"/content/nb09_{RUN_TAG}")RESTORE_DIR = WORK_DIR / "adapters"THETA_SFT   = WORK_DIR / "theta_sft" / "merged"ART_DIR     = WORK_DIR / "artifacts"REPORT      = ART_DIR / "report.json"BACKUP_DIR  = f"/content/drive/MyDrive/nb09_geometry_{RUN_TAG}"AXES = ["helpfulness", "correctness", "coherence", "complexity", "verbosity"]# --- precision: non-negotiable ----------------------------------------------MERGE_DTYPE = "float32"      # S9/S10 assert this; bf16 destroys the endpoint# --- geometry settings ------------------------------------------------------ORACLE_TOL      = 1e-6       # max rel. error between merge.py and the rank-space oracleFLOOR_SEED      = 137        # same seed as the PPO prompt stream, by conventionN_DIRICHLET     = 64# --- LMC settings -----------------------------------------------------------LMC_T_GRID   = [0.0, 0.25, 0.5, 0.75, 1.0]LMC_N_SEQ    = 64LMC_MAX_LEN  = 512LMC_BATCH    = 4LMC_SEED     = 137# --- gates: enable one at a time --------------------------------------------RUN_RESTORE   = TrueRUN_STATIC    = FalseRUN_GEOMETRY  = FalseRUN_FLOOR     = FalseRUN_LMC       = Falsefor d in (WORK_DIR, RESTORE_DIR, ART_DIR):    d.mkdir(parents=True, exist_ok=True)from google.colab import drivedrive.mount("/content/drive")os.makedirs(BACKUP_DIR, exist_ok=True)assert os.path.ismount("/content/drive"), "Drive not mounted — results would live on ephemeral disk"print(f"base        = {BASE_MODEL}")print(f"merge dtype = {MERGE_DTYPE}")print(f"work        = {WORK_DIR}")print(f"backup      = {BACKUP_DIR}")print(f"gates       = restore:{RUN_RESTORE} static:{RUN_STATIC} geom:{RUN_GEOMETRY} "      f"floor:{RUN_FLOOR} lmc:{RUN_LMC}")

**Cell 11**A tiny report helper. Every phase appends its own block and the file is rewritten atomically, so a crashed cell cannot leave a half-written report that the next gate then reads as a pass.

In [ ]:
# Cell 12import json, tempfiledef report_read():    if REPORT.exists():        return json.loads(REPORT.read_text())    return {}def report_write(phase, payload):    d = report_read()    d[phase] = payload    fd, tmp = tempfile.mkstemp(dir=str(ART_DIR)); os.close(fd)    Path(tmp).write_text(json.dumps(d, indent=2, sort_keys=True, default=str))    os.replace(tmp, REPORT)    print(f"[report] {phase}: {payload.get('status', '?')}")def gate_passed(phase):    return report_read().get(phase, {}).get("status") == "PASS"def require(phase):    assert gate_passed(phase), f"gate {phase!r} has not passed — do not enable the next flag"print("report helper ready:", REPORT)

**Cell 13**## 5. Restore the adapters from the NB08 bundleThe bundle is verified against its SHA256 sidecar **before** extraction, not after. A corrupted download that is unzipped first and checked later leaves a plausible-looking adapter tree on disk that someone will eventually use.After extraction each axis must contribute all three files: the adapter weights, the value head, and the PPO log. A missing `value_head.pt` is harmless for geometry but signals an incomplete bundle, and an incomplete bundle should not silently become the basis of the R that goes into the thesis.

In [ ]:
# Cell 14import hashlib, zipfileif RUN_RESTORE:    zip_path = Path(ADAPTER_ZIP)    assert zip_path.exists(), f"bundle not found: {zip_path}"    h = hashlib.sha256()    with open(zip_path, "rb") as f:        for chunk in iter(lambda: f.read(1 << 20), b""):            h.update(chunk)    digest = h.hexdigest()    print(f"bundle   {zip_path.name}  {zip_path.stat().st_size/1e6:.1f} MB")    print(f"sha256   {digest}")    sha_file = Path(ADAPTER_SHA_FILE)    if sha_file.exists():        expected = sha_file.read_text().split()[0].strip()        assert digest == expected, (            f"SHA256 MISMATCH\n  expected {expected}\n  actual   {digest}\n"            "The bundle is not the one NB08 produced. Stop.")        print("sha256   verified against sidecar")    else:        print(f"sha256   NO sidecar at {sha_file} — recording digest, chain is weaker")    with zipfile.ZipFile(zip_path) as z:        z.extractall(RESTORE_DIR)    need = ["adapter/adapter_model.safetensors", "value_head.pt", "ppo_log.json"]    missing, found = [], {}    for a in AXES:        for f in need:            p = RESTORE_DIR / f"ppo_{a}" / f            if not p.exists():                missing.append(f"{a}/{f}")        found[a] = str(RESTORE_DIR / f"ppo_{a}" / "adapter")    assert not missing, f"bundle incomplete: {missing}"    ADAPTER_PATHS = {a: Path(found[a]) for a in AXES}    for a in AXES:        print(f"  {a:12} {found[a]}")    report_write("restore", {        "status": "PASS", "zip": str(zip_path), "sha256": digest,        "sha256_verified": sha_file.exists(), "axes": AXES,        "adapter_paths": found,    })else:    ADAPTER_PATHS = {a: RESTORE_DIR / f"ppo_{a}" / "adapter" for a in AXES}    print("RUN_RESTORE off — reusing previously extracted adapters")

**Cell 15**## 6. Reconstruct θ_SFTθ_SFT is the **base shortcut**: an untouched fp32 snapshot of TinyLlama-1.1B-Chat-v1.0, no training of any kind. It is not shipped in the bundle because it is 4 GB that is fully recoverable from a model name — but "recoverable" only holds as long as the Hub serves the same weights, and the repo revision is not pinned by default.If `theta_sft_provenance.txt` exists it is used to pin the revision. If it does not, the current revision is recorded so a later run can detect a change. This matters because D, R and every downstream number are differences against this snapshot.

In [ ]:
# Cell 16import torchfrom transformers import AutoModelForCausalLM, AutoTokenizerREVISION = Noneprov = Path(THETA_SFT_PROV)if prov.exists():    for line in prov.read_text().splitlines():        if line.startswith("revision "):            REVISION = line.split()[1].strip()    print(f"revision pinned from provenance file: {REVISION}")if (THETA_SFT / "config.json").exists():    print(f"theta_SFT already present at {THETA_SFT}")else:    THETA_SFT.mkdir(parents=True, exist_ok=True)    kw = {"torch_dtype": torch.float32}    if REVISION:        kw["revision"] = REVISION    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)    m.save_pretrained(str(THETA_SFT))    AutoTokenizer.from_pretrained(BASE_MODEL, **({"revision": REVISION} if REVISION else {})                                  ).save_pretrained(str(THETA_SFT))    del m    print(f"theta_SFT = base snapshot written to {THETA_SFT} (fp32)")if REVISION is None:    try:        from huggingface_hub import HfApi        REVISION = HfApi().model_info(BASE_MODEL).sha        prov.write_text(f"{BASE_MODEL}\nrevision {REVISION}\ndtype float32\n")        print(f"revision recorded for future runs: {REVISION}")    except Exception as e:        print(f"could not resolve revision ({e}); provenance stays open")

**Cell 17**## 7. Static checks S9 / S10The NB08 verifier reported 16 of 18 checks; S9 and S10 never appeared in the output and the handoff instruction was not to treat that as a PASS. Both belong here, because both guard the path this notebook actually uses.- **S9 — `merge_linearity_verified` must not be hardcoded.** The flag has to be *derived* from a numerical comparison that can fail, not asserted in source. S9 greps the merge module for a literal assignment and refuses to pass if it finds one.- **S10 — no reduced precision anywhere in the evaluation path.** `MERGE_DTYPE` must resolve to `torch.float32`, and the merge module must not silently default to bf16.These run before any geometry. If either fails, nothing downstream is trustworthy.

In [ ]:
# Cell 18import re, inspectif RUN_STATIC:    require("restore")    import src.merge as M    src_text = inspect.getsource(M)    checks = {}    # --- S9 -----------------------------------------------------------------    hardcoded = re.findall(r"merge_linearity_verified\s*=\s*(True|1)\b", src_text)    checks["S9_no_hardcoded_linearity_flag"] = {        "pass": len(hardcoded) == 0,        "detail": f"{len(hardcoded)} literal assignment(s) found",    }    # --- S10 ----------------------------------------------------------------    resolved = M.resolve_torch_dtype(MERGE_DTYPE)    default  = M.resolve_torch_dtype(None)    checks["S10_merge_dtype_is_fp32"] = {        "pass": (MERGE_DTYPE == "float32" and resolved is torch.float32                 and default is torch.float32),        "detail": f"MERGE_DTYPE={MERGE_DTYPE} resolved={resolved} default={default}",    }    bad_default = re.search(r"dtype\s*[:=]\s*[\"']?(bf16|bfloat16|fp16|float16)", src_text)    checks["S10b_no_reduced_precision_default"] = {        "pass": bad_default is None,        "detail": "clean" if bad_default is None else f"found {bad_default.group(0)!r}",    }    for name, c in checks.items():        print(f"  {'PASS' if c['pass'] else 'FAIL'}  {name}: {c['detail']}")    ok = all(c["pass"] for c in checks.values())    report_write("static_checks", {        "status": "PASS" if ok else "FAIL", "checks": checks})    assert ok, "S9/S10 failed — do not compute geometry on this merge path"else:    print("RUN_STATIC off")

**Cell 19**## 8. D and R$D_i$ is the exact effective LoRA update of axis $i$, i.e. $\theta_i - \theta_{\mathrm{SFT}}$ summed over all seven target modules and all layers. $R = D^\top D$ is its Gram matrix; $R_{\cos}$ normalises it to unit diagonal.**Two independent paths, deliberately.**1. **Primary — `src/merge.py`.** `effective_deltas` materialises $s_i B_iA_i$ per module in fp32 and R is accumulated from the flattened inner products. This is the production code the rest of the project uses, so it is what must be tested.2. **Oracle — rank space.** For a single module, $\langle\delta_i,\delta_j\rangle = s_is_j\,\mathrm{tr}\!\left((B_i^\top B_j)(A_jA_i^\top)\right)$, where both factors are $r\times r$ with $r=8$. This never materialises the full update, uses a completely different arithmetic path, and is exact.Agreement between the two to `ORACLE_TOL` is what sets `merge_linearity_verified` — the flag S9 refuses to see hardcoded.**And a control that must fail.** A test that cannot fail proves nothing. Cell 20 also computes the PEFT-linear cross-term contamination on the same adapters. If that control comes out *within* tolerance, the oracle is not sensitive enough to detect the very error it exists to catch, and the whole check is void regardless of what the primary comparison says.Pairs are streamed: at most two adapters are resident at any moment.

In [ ]:
# Cell 20import numpy as np, gc, itertools, timeif RUN_GEOMETRY:    require("static_checks")    from src.merge import effective_deltas    from src.effective_lora_geometry import load_effective_lora_geometry    m = len(AXES)    R_primary = np.zeros((m, m), dtype=np.float64)    R_oracle  = np.zeros((m, m), dtype=np.float64)    R_peft    = np.zeros((m, m), dtype=np.float64)   # contaminated control    GEO = {a: load_effective_lora_geometry(Path(ADAPTER_PATHS[a])) for a in AXES}    modules = sorted(GEO[AXES[0]].keys())    print(f"{len(modules)} LoRA target modules per adapter")    def inner_oracle(a, b):        # <d_a, d_b> in rank space: tr((Ba^T Bb)(Ab Aa^T)) * s_a * s_b        tot = 0.0        for mn in modules:            la, lb = GEO[a][mn], GEO[b][mn]            Ba = la.lora_b.detach().to(torch.float64)            Aa = la.lora_a.detach().to(torch.float64)            Bb = lb.lora_b.detach().to(torch.float64)            Ab = lb.lora_a.detach().to(torch.float64)            P = Ba.T @ Bb                       # r x r            Q = Ab @ Aa.T                       # r x r            tot += float((P * Q.T).sum()) * float(la.scaling) * float(lb.scaling)        return tot    def inner_peft_control(a, b):        # one of the spurious Ba@Ab cross terms PEFT's linear path introduces        tot = 0.0        for mn in modules:            la, lb = GEO[a][mn], GEO[b][mn]            Ba = la.lora_b.detach().to(torch.float64); Aa = la.lora_a.detach().to(torch.float64)            Bb = lb.lora_b.detach().to(torch.float64); Ab = lb.lora_a.detach().to(torch.float64)            sa, sb = float(la.scaling), float(lb.scaling)            d_a = sa * (Ba @ Aa)            cross = np.sqrt(sa * sb) * (Ba @ Ab)     # one of the spurious terms            tot += float((d_a * cross).sum())        return tot    t0 = time.time()    for i, j in itertools.combinations_with_replacement(range(m), 2):        ai, aj = AXES[i], AXES[j]        paths = {ai: Path(ADAPTER_PATHS[ai])} if i == j else \                {ai: Path(ADAPTER_PATHS[ai]), aj: Path(ADAPTER_PATHS[aj])}        dl = effective_deltas(paths)        val = 0.0        for mn in modules:            val += float((dl[ai][mn].to(torch.float64) * dl[aj][mn].to(torch.float64)).sum())        R_primary[i, j] = R_primary[j, i] = val        R_oracle[i, j]  = R_oracle[j, i]  = inner_oracle(ai, aj)        R_peft[i, j]    = R_peft[j, i]    = inner_peft_control(ai, aj)        del dl; gc.collect()        print(f"  ({ai[:6]},{aj[:6]}) primary={val:.6e}  oracle={R_oracle[i,j]:.6e}")    scale = np.abs(R_primary).max()    rel_err     = np.abs(R_primary - R_oracle).max() / scale    ctrl_relerr = np.abs(R_peft - R_oracle).max() / scale    print(f"\nelapsed {time.time()-t0:.0f}s")    print(f"max rel. error  primary vs oracle : {rel_err:.3e}   (tol {ORACLE_TOL:.0e})")    print(f"max rel. error  PEFT control      : {ctrl_relerr:.3e}   (must exceed tol)")    merge_linearity_verified = bool(rel_err < ORACLE_TOL)    control_has_teeth        = bool(ctrl_relerr > ORACLE_TOL)    d = np.sqrt(np.diag(R_primary))    R_cos = R_primary / np.outer(d, d)    eigvals = np.linalg.eigvalsh(R_primary)    w, V = np.linalg.eigh(R_primary)    perron = np.abs(V[:, -1]); perron = perron / perron.sum()    np.save(ART_DIR / "R_gram.npy", R_primary)    np.save(ART_DIR / "R_cos.npy", R_cos)    np.save(ART_DIR / "D_norms.npy", d)    print("\nR_cos:")    print("            " + "".join(f"{a[:6]:>9}" for a in AXES))    for i, a in enumerate(AXES):        print(f"{a:12}" + "".join(f"{R_cos[i,k]:9.3f}" for k in range(m)))    print(f"\n||D_i||   : {np.array2string(d, precision=4)}")    print(f"eigenvalues: {np.array2string(eigvals, precision=4)}")    print(f"Perron     : {np.array2string(perron, precision=4)}")    ok = merge_linearity_verified and control_has_teeth    report_write("geometry", {        "status": "PASS" if ok else "FAIL",        "merge_linearity_verified": merge_linearity_verified,        "control_has_teeth": control_has_teeth,        "oracle_max_rel_err": rel_err,        "peft_control_rel_err": ctrl_relerr,        "oracle_tol": ORACLE_TOL,        "axes": AXES,        "D_norms": d.tolist(),        "R_gram": R_primary.tolist(),        "R_cos": R_cos.tolist(),        "eigenvalues": eigvals.tolist(),        "perron": perron.tolist(),        "n_modules": len(modules),    })    assert merge_linearity_verified, "merge.py disagrees with the oracle — do not proceed"    assert control_has_teeth, "PEFT control passed the tolerance — the check cannot fail, so it proves nothing"else:    print("RUN_GEOMETRY off")

**Cell 21**## 9. Deliverable (b) — floor LP and $R^-$The improvement floor is $\mathcal F_p=\{\lambda\in\Delta^{m-1}: \big(R(\lambda-p)\big)_i\ge 0\ \forall i\}$: coefficient vectors that, under the R proxy, improve **no** objective's expectation at the cost of another. The question is whether it contains anything besides $p$ itself.The criterion is the **LP value, not the sign pattern of $R$**. Negative off-diagonals are neither necessary nor sufficient — empirically only about 43 % of PSD matrices with negative off-diagonals have an open floor. So:$$t^\star(p)=\max_{\lambda\in\Delta^{m-1}} \; t \quad \text{s.t.}\quad R(\lambda-p)\ \ge\ t\mathbf 1 .$$$\lambda=p$ is always feasible with $t=0$, so $t^\star\ge 0$ always. $t^\star>0$ means a strictly common improvement exists and the floor-projection family has somewhere to go; $t^\star=0$ means the floor collapses to $p$ and every floor method returns $\lambda\approx p$ (Prop 26).$R^-=\min(R,0)$ elementwise and $C(\lambda)=\lambda^\top R^-\lambda$ are reported alongside as the *diagnostic* — they say whether any conflict structure exists at all, but they do not decide the floor.Evaluated over a search set: the five vertices, the uniform point, and 64 Dirichlet(1) draws at seed 137. The 13 pre-registered evaluation preferences should be added from the pre-registration file to reach the full |B| = 77 before anything is reported as final.

In [ ]:
# Cell 22from scipy.optimize import linprogif RUN_FLOOR:    require("geometry")    R = np.load(ART_DIR / "R_gram.npy")    m = R.shape[0]    rng = np.random.default_rng(FLOOR_SEED)    P = [np.eye(m)[i] for i in range(m)] + [np.full(m, 1.0 / m)]    names = list(AXES) + ["uniform"]    for k in range(N_DIRICHLET):        P.append(rng.dirichlet(np.ones(m)))        names.append(f"dirichlet_{k:02d}")    def floor_lp(R, p):        # max t  s.t.  R(lam-p) >= t*1,  sum lam = 1,  lam >= 0        c = np.concatenate([np.zeros(m), [-1.0]])        A_ub = np.hstack([-R, np.ones((m, 1))])        b_ub = -R @ p        A_eq = np.concatenate([np.ones(m), [0.0]])[None, :]        bounds = [(0.0, 1.0)] * m + [(None, None)]        res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1.0],                      bounds=bounds, method="highs")        if not res.success:            return np.nan, None        return float(res.x[-1]), res.x[:m]    R_neg = np.minimum(R, 0.0)    n_neg = int((R_neg < 0).sum())    print(f"R- : {n_neg} negative entries, max magnitude {np.abs(R_neg).max():.6e}")    rows, t_vals, c_vals = [], [], []    for nm, p in zip(names, P):        t, lam = floor_lp(R, p)        C = float(p @ R_neg @ p)        t_vals.append(t); c_vals.append(C)        moved = np.nan if lam is None else float(np.abs(lam - p).max())        rows.append({"p": nm, "t_star": t, "C": C, "max_shift": moved})    t_arr = np.array(t_vals)    n_open = int((t_arr > 1e-9).sum())    print(f"\nfloor open on {n_open}/{len(t_arr)} preferences   "          f"t* max={np.nanmax(t_arr):.3e}  median={np.nanmedian(t_arr):.3e}")    print(f"C(lambda) max |.| = {np.abs(np.array(c_vals)).max():.3e}")    print("\nvertices + uniform:")    for r in rows[:m + 1]:        print(f"  {r['p']:12} t*={r['t_star']:.3e}  C={r['C']:.3e}  max|lam-p|={r['max_shift']:.3e}")    verdict = "FLOOR_OPEN" if n_open > 0 else "FLOOR_COLLAPSED"    print(f"\nverdict: {verdict}")    if verdict == "FLOOR_COLLAPSED":        print("  -> every floor-projection method returns lambda ~ p (Prop 26 class result)")    else:        print("  -> the floor family is alive on this R; v5 collapse does NOT reproduce")    report_write("floor", {        "status": "PASS", "verdict": verdict,        "n_open": n_open, "n_preferences": len(t_arr),        "t_star_max": float(np.nanmax(t_arr)), "t_star_median": float(np.nanmedian(t_arr)),        "R_neg_count": n_neg, "R_neg_max_abs": float(np.abs(R_neg).max()),        "C_max_abs": float(np.abs(np.array(c_vals)).max()),        "seed": FLOOR_SEED, "n_dirichlet": N_DIRICHLET,        "search_set_note": "vertices + uniform + Dirichlet; 13 pre-registered eval preferences NOT yet included",        "rows": rows,    })else:    print("RUN_FLOOR off")

**Cell 23**## 10. Deliverable (d) — linear mode connectivityLMC asks whether the interpolation family is a sensible object at all: if the loss spikes between two endpoints, a merged model at an interior $\lambda$ is not "between" the specialists in any useful sense, and coefficient-space reasoning about $\theta(\lambda)$ loses its footing.**Reward-free by construction.** The metric is the model's own token-level negative log-likelihood on held-out HelpSteer2 responses — no ArmoRM, no reward query. This is what keeps LMC on the unblocked side of the §6 escalation.For each path the barrier is$$B=\max_t\Big[L(t)-\big((1-t)L(0)+tL(1)\big)\Big],$$i.e. how far the measured loss rises above the straight line joining the endpoints. $B\le 0$ means no barrier.Paths measured: the ten vertex-to-vertex edges $\lambda=(1-t)e_i+te_j$, which lie inside the simplex and are the ones merging actually traverses.**One implementation detail that matters.** `apply_effective_deltas_to_model` writes $\theta_0+\delta$ in place, so calling it twice compounds. The pristine target weights are cached once and restored before every merge; without that, each successive λ would be measured on top of the previous one.

In [ ]:
# Cell 24if RUN_LMC:    require("floor")    from datasets import load_dataset    from src.merge import effective_deltas, combine_effective_deltas, resolve_base_module    assert torch.cuda.is_available(), "LMC needs a GPU"    tok = AutoTokenizer.from_pretrained(str(THETA_SFT))    if tok.pad_token is None:        tok.pad_token = tok.eos_token    ds = load_dataset("nvidia/HelpSteer2", split="validation")    ds = ds.shuffle(seed=LMC_SEED).select(range(LMC_N_SEQ))    def encode(ex):        msgs = [{"role": "user", "content": ex["prompt"]},                {"role": "assistant", "content": ex["response"]}]        try:            text = tok.apply_chat_template(msgs, tokenize=False)        except Exception:            text = f"<|user|>\n{ex['prompt']}</s>\n<|assistant|>\n{ex['response']}</s>"        return text    texts = [encode(ex) for ex in ds]    enc = tok(texts, return_tensors="pt", padding=True, truncation=True,              max_length=LMC_MAX_LEN)    print(f"{len(texts)} held-out sequences, max_len {enc['input_ids'].shape[1]}")    model = AutoModelForCausalLM.from_pretrained(        str(THETA_SFT), torch_dtype=torch.float32).to("cuda").eval()    assert next(model.parameters()).dtype is torch.float32, "S10: eval path must be fp32"    @torch.no_grad()    def mean_nll():        tot, ntok = 0.0, 0        for s in range(0, len(texts), LMC_BATCH):            ids = enc["input_ids"][s:s + LMC_BATCH].to("cuda")            att = enc["attention_mask"][s:s + LMC_BATCH].to("cuda")            labels = ids.clone(); labels[att == 0] = -100            out = model(input_ids=ids, attention_mask=att, labels=labels)            n = int((labels[:, 1:] != -100).sum())            tot += float(out.loss) * n; ntok += n        return tot / ntok    lmc_rows = []    for i, j in itertools.combinations(range(len(AXES)), 2):        ai, aj = AXES[i], AXES[j]        DL = effective_deltas({ai: Path(ADAPTER_PATHS[ai]), aj: Path(ADAPTER_PATHS[aj])})        mods = sorted(DL[ai].keys())        handles = {mn: resolve_base_module(model, mn) for mn in mods}        cache = {mn: handles[mn].weight.detach().clone().cpu() for mn in mods}        losses = []        for t in LMC_T_GRID:            merged = combine_effective_deltas([1.0 - t, t], DL)            with torch.no_grad():                for mn in mods:                    w = handles[mn].weight                    w.copy_((cache[mn] + merged[mn]).to(w.device, torch.float32))            losses.append(mean_nll())            del merged        # restore pristine weights before the next pair        with torch.no_grad():            for mn in mods:                handles[mn].weight.copy_(cache[mn].to(handles[mn].weight.device))        del DL, cache; gc.collect(); torch.cuda.empty_cache()        L = np.array(losses); ts = np.array(LMC_T_GRID)        chord = L[0] + (L[-1] - L[0]) * ts        barrier = float((L - chord).max())        lmc_rows.append({"pair": f"{ai}|{aj}", "t_grid": LMC_T_GRID,                         "loss": L.tolist(), "barrier": barrier})        print(f"  {ai[:6]:>6}-{aj[:6]:<6} L={np.array2string(L, precision=4)}  barrier={barrier:+.4f}")    barriers = np.array([r["barrier"] for r in lmc_rows])    print(f"\nmax barrier {barriers.max():+.4f}   mean {barriers.mean():+.4f}   "          f"n>0: {(barriers > 0).sum()}/{len(barriers)}")    verdict = "LMC_CONNECTED" if barriers.max() <= 0 else "BARRIER_PRESENT"    print(f"verdict: {verdict}")    report_write("lmc", {        "status": "PASS", "verdict": verdict,        "max_barrier": float(barriers.max()), "mean_barrier": float(barriers.mean()),        "n_positive": int((barriers > 0).sum()), "n_paths": len(lmc_rows),        "n_sequences": LMC_N_SEQ, "max_len": LMC_MAX_LEN, "seed": LMC_SEED,        "metric": "mean token NLL on held-out HelpSteer2 validation responses (no ArmoRM)",        "rows": lmc_rows,    })    del model; gc.collect(); torch.cuda.empty_cache()else:    print("RUN_LMC off")

**Cell 25**## 11. Bundle the resultsSame discipline as NB08: artifacts plus `report.json` plus a SHA256 manifest, copied to Drive. The manifest is what lets a later notebook — or a reader — confirm that the R a claim rests on is the R this run produced.

In [ ]:
# Cell 26import hashlib, shutild = report_read()print("gate status:")for k in ["restore", "static_checks", "geometry", "floor", "lmc"]:    print(f"  {k:15} {d.get(k, {}).get('status', '-')}")manifest = []for f in sorted(ART_DIR.rglob("*")):    if f.is_file() and f.name != "manifest.sha256":        h = hashlib.sha256(f.read_bytes()).hexdigest()        manifest.append(f"{h}  {f.relative_to(ART_DIR)}")(ART_DIR / "manifest.sha256").write_text("\n".join(manifest) + "\n")print(f"\n{len(manifest)} artifacts hashed")out_zip = f"/content/nb09_geometry_{RUN_TAG}.zip"shutil.make_archive(out_zip[:-4], "zip", root_dir=str(ART_DIR))zh = hashlib.sha256(open(out_zip, "rb").read()).hexdigest()shutil.copy(out_zip, f"{BACKUP_DIR}/nb09_geometry_{RUN_TAG}.zip")Path(f"{BACKUP_DIR}/nb09_geometry_{RUN_TAG}.sha256").write_text(    f"{zh}  nb09_geometry_{RUN_TAG}.zip\n")print(f"{os.path.getsize(out_zip)/1e6:.2f} MB -> {BACKUP_DIR}")print(f"sha256 {zh}")

**Cell 27**## 12. What comes nextDeliverables (b) and (d) are now on the PPO adapters rather than the v5 SFT-R, and every number in the thesis that came from the old R should be replaced from `report.json` and labelled with this run.Still open before anything here is final:1. **The 13 pre-registered evaluation preferences** are not yet in the search set — the floor result is over 70 of the 77 points. Load them from the pre-registration and rerun Cell 22.2. **Method outputs on the new R** — Avg, MaxMin(c), Cert, Fair(α,ε) — can be computed geometrically from `R_gram.npy` without touching ArmoRM: which λ each returns, whether it moves off p, whether the guarantee is non-empty. That is still deliverable (b) territory.3. **Deliverable (a) stays blocked.** The vertex matrix M, $U_p(\lambda)$ and the method comparison need ArmoRM and hang on the §6 escalation. Nothing in this notebook should be read as evidence about them.